# Taller 2: ETL con Apache Spark y Delta Lake en Databricks

En este taller construirás un flujo ETL completo sobre datos de ventas aplicando la **arquitectura Medallion** (Bronze → Silver → Gold) con **Apache Spark** y **Delta Lake** como formato de tabla abierto.

## Objetivos de aprendizaje
- Configurar Databricks Community Edition desde cero
- Implementar la arquitectura Lakehouse Medallion
- Leer, transformar y escribir datos con la API de DataFrames de PySpark
- Usar Delta Lake: transacciones ACID, versionado y time travel
- Ejecutar consultas analíticas sobre tablas Gold

## Contenido
0. Configuración del entorno (Databricks CE)
1. Bronze — Ingesta de datos crudos
2. Silver — Limpieza y transformación
3. Gold — Métricas y agregaciones
4. Delta Lake — Capacidades avanzadas
5. Ejercicios integradores

---
## 0. Configuración del entorno

Sigue estos pasos **una sola vez** antes de ejecutar el notebook.

---

### Paso 1 — Crear una cuenta en Databricks Community Edition

Databricks ofrece una capa gratuita llamada **Community Edition** que no requiere tarjeta de crédito.

1. Ve a https://www.databricks.com/try-databricks
2. Haz clic en **"Get started for free"** o **"Try Databricks"**
3. Completa el formulario de registro (nombre, correo, empresa/universidad)
4. En la pantalla de selección de plan, elige **"Community Edition"** (opción gratuita)
5. Verifica tu correo electrónico y accede a https://community.cloud.databricks.com

> **Limitaciones de Community Edition:** un solo cluster de 1 nodo, se apaga automáticamente tras 2 horas de inactividad. Es suficiente para este taller.

---

### Paso 2 — Crear un cluster

El cluster es el motor de cómputo donde correrá Spark.

1. En la barra lateral izquierda, haz clic en **Compute**
2. Haz clic en **"Create compute"** (o **"+ New cluster"**)
3. Completa los campos:
   - **Cluster name:** `taller2-cluster` (o cualquier nombre)
   - **Databricks Runtime:** elige la versión más reciente con etiqueta **LTS** (ej. `14.x LTS` o superior)
   - Las demás opciones se configuran solas en Community Edition
4. Haz clic en **"Create compute"**
5. Espera 3-5 minutos hasta que el estado cambie a **Running** (círculo verde)

> El runtime LTS incluye Apache Spark y Delta Lake preinstalados. No necesitas instalar nada adicional.

---

### Paso 3 — Importar este notebook en Databricks

**Opción A — Importar desde archivo .ipynb:**
1. En la barra lateral, ve a **Workspace** → tu carpeta de usuario (ícono de persona)
2. Haz clic derecho → **Import**
3. Selecciona **"File"**, arrastra o selecciona el archivo `taller_databricks_etl.ipynb`
4. Haz clic en **Import**

**Opción B — Crear notebook nuevo y pegar el código:**
1. Ve a **Workspace** → **Create** → **Notebook**
2. Nómbralo `taller_databricks_etl`, elige Python como lenguaje
3. Copia y pega cada celda de código manualmente

---

### Paso 4 — Conectar el notebook al cluster

1. Abre el notebook en Databricks
2. En la parte superior derecha verás un menú desplegable que dice **"Detached"** o **"Connect"**
3. Haz clic → selecciona el cluster `taller2-cluster` que creaste
4. El estado cambia a **"Attached"** o muestra el nombre del cluster

> En Databricks, `spark` y `dbutils` están disponibles automáticamente. No necesitas importar ni inicializar `SparkSession`.

---

### Paso 5 — Ejecutar las celdas

- Ejecutar celda actual: `Shift + Enter`
- Ejecutar todo el notebook: menú **Run** → **Run all**
- Ejecutar desde la celda actual: menú **Run** → **Run all below**

Ejecuta las celdas **en orden de arriba hacia abajo** la primera vez.

---
## Arquitectura del taller

```
  FUENTE DE DATOS
  (datos sintéticos)
        |
        v
 +--------------+
 |    BRONZE    |  Datos crudos tal como llegan
 |  Delta Lake  |  Sin transformaciones
 +--------------+
        |
        v
 +--------------+
 |    SILVER    |  Datos limpios y enriquecidos
 |  Delta Lake  |  Nulos resueltos, tipos correctos,
 +--------------+  columnas derivadas
        |
        v
 +--------------+
 |     GOLD     |  Métricas y agregaciones
 |  Delta Lake  |  Listas para consumo analítico
 +--------------+
```

**Formato de almacenamiento:** Delta Lake (formato de tabla abierto basado en Parquet + transaction log)

**Ubicación:** DBFS (Databricks File System) en la ruta `/tmp/taller2/`

In [ ]:
# =============================================================
# CONFIGURACIÓN DEL ENTORNO
# Ejecuta esta celda primero. Define las rutas base en DBFS.
# =============================================================

# Rutas base en el sistema de archivos de Databricks (DBFS)
BASE_PATH   = "/tmp/taller2"
BRONZE_PATH = f"{BASE_PATH}/bronze"
SILVER_PATH = f"{BASE_PATH}/silver"
GOLD_PATH   = f"{BASE_PATH}/gold"

# Limpiar ejecuciones anteriores para partir de cero
dbutils.fs.rm(BASE_PATH, recurse=True)
print("Directorio anterior eliminado (si existia).")

print(f"\nRutas configuradas:")
print(f"  Bronze : {BRONZE_PATH}")
print(f"  Silver : {SILVER_PATH}")
print(f"  Gold   : {GOLD_PATH}")
print(f"\nSpark version: {spark.version}")

---
## Parte 1 — Bronze: Ingesta de datos crudos

La capa Bronze almacena los datos **exactamente como llegan** desde la fuente, sin ninguna transformación.  
Su propósito es preservar el dato original para auditoría y reprocesamiento.

En este taller generaremos datos sintéticos de transacciones de ventas que simulan la llegada de un archivo CSV externo.  
Los datos incluyen **problemas de calidad intencionales** que corregiremos en Silver:
- ~10% de valores nulos en `discount_pct`
- ~3% de precios inválidos (valores negativos)

In [ ]:
# =============================================================
# 1.1 — GENERAR DATOS SINTÉTICOS DE VENTAS
# Usamos funciones nativas de Spark para generar 1.000 filas.
# Esto simula la llegada de un archivo externo (CSV, JSON, etc.)
# =============================================================

from pyspark.sql import functions as F

SEED = 42  # semilla para reproducibilidad

# Listas de valores posibles
arr_categorias = F.array([F.lit(c) for c in ["Electronica", "Muebles", "Ropa", "Alimentos", "Libros"]])
arr_ciudades   = F.array([F.lit(c) for c in ["Bogota", "Medellin", "Cali", "Barranquilla", "Cartagena"]])
arr_pagos      = F.array([F.lit(p) for p in ["tarjeta_credito", "tarjeta_debito", "efectivo", "transferencia"]])
arr_productos  = F.array([F.lit(p) for p in [
    "Laptop Pro", "Silla Ergonomica", "Camiseta Algodon", "Arroz Premium",
    "Python Avanzado", "Monitor 4K", "Escritorio Pie", "Jean Clasico",
    "Aceite Oliva", "SQL para Todos", "Auriculares BT", "Estante Madera",
    "Zapatos Cuero", "Leche Entera", "Historia IA"
]])

# Generamos 1.000 filas con spark.range y columnas aleatorias
df_bronze_raw = (
    spark.range(1, 1001)
    .select(
        F.col("id"),
        F.rand(SEED).alias("r1"),
        F.rand(SEED + 1).alias("r2"),
        F.rand(SEED + 2).alias("r3"),
        F.rand(SEED + 3).alias("r4"),
        F.rand(SEED + 4).alias("r5"),
        F.rand(SEED + 5).alias("r6"),
        F.rand(SEED + 6).alias("r7"),
    )
    .select(
        # ID de transacción: TXN-00001 ... TXN-01000
        F.concat(F.lit("TXN-"), F.lpad(F.col("id").cast("string"), 5, "0")).alias("transaction_id"),
        # ID cliente: C0001 ... C0200
        F.concat(F.lit("C"), F.lpad(((F.col("r1") * 200).cast("int") + 1).cast("string"), 4, "0")).alias("customer_id"),
        # Producto y categoría (de las listas)
        arr_productos[(F.col("r2") * 15).cast("int")].alias("product_name"),
        arr_categorias[(F.col("r3") * 5).cast("int")].alias("category"),
        # Cantidad entre 1 y 10
        ((F.col("r4") * 9).cast("int") + 1).alias("quantity"),
        # Precio: ~3% de valores -99 (inválidos) para limpiar en Silver
        F.when(F.col("r5") < 0.03, F.lit(-99.0))
         .otherwise(F.round(F.col("r5") * 1995 + 5, 2)).alias("unit_price"),
        # Fecha entre 2024-01-01 y 2024-12-31
        F.date_add(F.lit("2024-01-01").cast("date"), (F.col("r6") * 365).cast("int")).alias("transaction_date"),
        arr_ciudades[(F.col("r1") * 5).cast("int")].alias("store_city"),
        arr_pagos[(F.col("r2") * 4).cast("int")].alias("payment_method"),
        # Descuento: ~10% de nulos para limpiar en Silver
        F.when(F.col("r7") > 0.9, F.lit(None).cast("double"))
         .otherwise((F.col("r4") * 4).cast("int") * 0.05).alias("discount_pct"),
    )
)

print(f"Registros generados: {df_bronze_raw.count():,}")
print("\nEsquema:")
df_bronze_raw.printSchema()
df_bronze_raw.show(5, truncate=False)

In [ ]:
# =============================================================
# 1.2 — GUARDAR BRONZE COMO DELTA LAKE
# Delta Lake = Parquet + transaction log (_delta_log/)
# Provee: transacciones ACID, versionado, time travel
# =============================================================

(
    df_bronze_raw
    .write
    .format("delta")          # formato Delta Lake
    .mode("overwrite")        # sobreescribir si existe
    .save(f"{BRONZE_PATH}/ventas/")
)

print(f"Tabla Bronze guardada en: {BRONZE_PATH}/ventas/")

# Ver los archivos generados (Parquet + transaction log)
print("\nArchivos en Bronze:")
display(dbutils.fs.ls(f"{BRONZE_PATH}/ventas/"))

In [ ]:
# =============================================================
# 1.3 — EXPLORAR LA CAPA BRONZE
# Verificamos los datos antes de pasar a Silver
# =============================================================

df_bronze = spark.read.format("delta").load(f"{BRONZE_PATH}/ventas/")

print("=== Estadísticas básicas de Bronze ===")
print(f"Total de registros : {df_bronze.count():,}")
print(f"Columnas           : {len(df_bronze.columns)}")

print("\n=== Conteo de nulos por columna ===")
df_bronze.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df_bronze.columns
]).show()

print("=== Precios invalidos (unit_price < 0) ===")
print(f"Registros con precio invalido: {df_bronze.filter(F.col('unit_price') < 0).count()}")

print("\n=== Distribucion por categoria ===")
df_bronze.groupBy("category").count().orderBy("count", ascending=False).show()

print("=== Rango de fechas ===")
df_bronze.select(
    F.min("transaction_date").alias("fecha_min"),
    F.max("transaction_date").alias("fecha_max")
).show()

---
## Parte 2 — Silver: Limpieza y transformación

La capa Silver aplica reglas de **calidad de datos** y **enriquecimiento**:

| Problema encontrado en Bronze | Acción en Silver |
|---|---|
| `unit_price` negativo (~3%) | Eliminar el registro |
| `discount_pct` nulo (~10%) | Rellenar con `0.0` (sin descuento) |
| Ausencia de columnas derivadas | Calcular `total_bruto`, `total_descuento`, `total_neto` |
| Sin dimensiones temporales | Extraer `year`, `month`, `month_name` |
| Sin auditoría | Agregar `processed_at` (timestamp de procesamiento) |

In [ ]:
# =============================================================
# 2.1 — TRANSFORMACIÓN SILVER
# Leer Bronze → limpiar → enriquecer → guardar Silver
# =============================================================

from pyspark.sql import functions as F

df_bronze = spark.read.format("delta").load(f"{BRONZE_PATH}/ventas/")

df_silver = (
    df_bronze

    # --- LIMPIEZA ---
    # 1. Eliminar registros con precio inválido
    .filter(F.col("unit_price") > 0)
    # 2. Rellenar nulos en discount_pct con 0 (sin descuento)
    .fillna({"discount_pct": 0.0})

    # --- ENRIQUECIMIENTO ---
    # 3. Calcular totales de la transacción
    .withColumn("total_bruto",
        F.round(F.col("quantity") * F.col("unit_price"), 2))
    .withColumn("total_descuento",
        F.round(F.col("total_bruto") * F.col("discount_pct"), 2))
    .withColumn("total_neto",
        F.round(F.col("total_bruto") - F.col("total_descuento"), 2))

    # 4. Extraer dimensiones de fecha
    .withColumn("year",       F.year("transaction_date"))
    .withColumn("month",      F.month("transaction_date"))
    .withColumn("month_name", F.date_format("transaction_date", "MMMM"))
    .withColumn("day_of_week",F.date_format("transaction_date", "EEEE"))

    # 5. Timestamp de procesamiento (auditoría)
    .withColumn("processed_at", F.current_timestamp())
)

registros_bronze = df_bronze.count()
registros_silver = df_silver.count()

print(f"Registros Bronze  : {registros_bronze:,}")
print(f"Registros Silver  : {registros_silver:,}")
print(f"Eliminados        : {registros_bronze - registros_silver:,} (precios invalidos)")

print("\nEsquema Silver:")
df_silver.printSchema()

In [ ]:
# =============================================================
# 2.2 — GUARDAR Y VERIFICAR SILVER
# =============================================================

# Guardar como Delta Table, particionado por año y mes
# El particionado acelera consultas que filtran por fecha
(
    df_silver
    .write
    .format("delta")
    .mode("overwrite")
    .partitionBy("year", "month")   # particionado físico en DBFS
    .save(f"{SILVER_PATH}/ventas/")
)

print(f"Tabla Silver guardada en: {SILVER_PATH}/ventas/")

# Verificar: no deben quedar nulos en columnas críticas
df_silver_check = spark.read.format("delta").load(f"{SILVER_PATH}/ventas/")

print("\n=== Verificacion de nulos en Silver (deben ser 0) ===")
df_silver_check.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in ["unit_price", "discount_pct", "total_neto"]
]).show()

print("=== Muestra de datos Silver ===")
display(df_silver_check.select(
    "transaction_id", "customer_id", "product_name", "category",
    "quantity", "unit_price", "discount_pct", "total_neto",
    "transaction_date", "year", "month"
).limit(10))

---
## Parte 3 — Gold: Métricas y agregaciones

La capa Gold contiene tablas **pre-agregadas** listas para consumo analítico (dashboards, reportes, ML).  
Cada tabla Gold responde una pregunta de negocio específica.

Crearemos tres tablas Gold:
| Tabla | Pregunta de negocio |
|---|---|
| `ventas_categoria_mes` | ¿Cuáles categorías generan más ingresos por mes? |
| `top_clientes` | ¿Quiénes son los clientes de mayor valor? |
| `metodos_pago` | ¿Qué método de pago es más usado y cuál genera más descuento? |

In [ ]:
# =============================================================
# 3.1 — GOLD: VENTAS POR CATEGORÍA Y MES
# =============================================================

from pyspark.sql import functions as F

df_silver = spark.read.format("delta").load(f"{SILVER_PATH}/ventas/")

gold_cat_mes = (
    df_silver
    .groupBy("year", "month", "month_name", "category")
    .agg(
        F.count("transaction_id").alias("num_transacciones"),
        F.sum("quantity").alias("unidades_vendidas"),
        F.round(F.sum("total_bruto"), 2).alias("ingresos_brutos"),
        F.round(F.sum("total_descuento"), 2).alias("total_descuentos"),
        F.round(F.sum("total_neto"), 2).alias("ingresos_netos"),
        F.round(F.avg("unit_price"), 2).alias("precio_promedio"),
    )
    .orderBy("year", "month", F.col("ingresos_netos").desc())
)

print("=== Ventas por categoria y mes (primeros 15 registros) ===")
display(gold_cat_mes.limit(15))

# Guardar como Delta
gold_cat_mes.write.format("delta").mode("overwrite").save(f"{GOLD_PATH}/ventas_categoria_mes/")
print(f"\nGuardado en: {GOLD_PATH}/ventas_categoria_mes/")

In [ ]:
# =============================================================
# 3.2 — GOLD: TOP CLIENTES POR GASTO TOTAL
# =============================================================

gold_top_clientes = (
    df_silver
    .groupBy("customer_id")
    .agg(
        F.count("transaction_id").alias("num_compras"),
        F.round(F.sum("total_neto"), 2).alias("gasto_total"),
        F.round(F.avg("total_neto"), 2).alias("gasto_promedio"),
        F.round(F.sum("quantity"), 0).alias("unidades_compradas"),
        F.countDistinct("category").alias("categorias_distintas"),
    )
    .withColumn(
        "segmento",
        F.when(F.col("gasto_total") >= 5000, "VIP")
         .when(F.col("gasto_total") >= 2000, "Premium")
         .when(F.col("gasto_total") >= 500,  "Regular")
         .otherwise("Ocasional")
    )
    .orderBy(F.col("gasto_total").desc())
)

print("=== Top 15 clientes por gasto total ===")
display(gold_top_clientes.limit(15))

print("\n=== Distribucion de clientes por segmento ===")
gold_top_clientes.groupBy("segmento").count().orderBy("count", ascending=False).show()

# Guardar como Delta
gold_top_clientes.write.format("delta").mode("overwrite").save(f"{GOLD_PATH}/top_clientes/")
print(f"Guardado en: {GOLD_PATH}/top_clientes/")

In [ ]:
# =============================================================
# 3.3 — GOLD: ANÁLISIS POR MÉTODO DE PAGO
# También usamos Spark SQL como alternativa a la API de DataFrames
# =============================================================

# Registrar la tabla Silver como vista temporal para usar SQL
df_silver.createOrReplaceTempView("silver_ventas")

gold_pagos = spark.sql("""
    SELECT
        payment_method,
        COUNT(*)                                         AS num_transacciones,
        ROUND(SUM(total_neto), 2)                        AS ingresos_netos,
        ROUND(AVG(total_neto), 2)                        AS ticket_promedio,
        ROUND(AVG(discount_pct) * 100, 1)               AS descuento_promedio_pct,
        ROUND(SUM(total_descuento), 2)                   AS total_descuentos_otorgados
    FROM silver_ventas
    GROUP BY payment_method
    ORDER BY ingresos_netos DESC
""")

print("=== Analisis por metodo de pago ===")
display(gold_pagos)

# Guardar como Delta
gold_pagos.write.format("delta").mode("overwrite").save(f"{GOLD_PATH}/metodos_pago/")
print(f"Guardado en: {GOLD_PATH}/metodos_pago/")

---
## Parte 4 — Delta Lake: Capacidades avanzadas

Delta Lake es un **formato de tabla abierto** construido sobre Parquet que agrega:

| Capacidad | Qué significa |
|---|---|
| **ACID Transactions** | Las escrituras son atómicas (todo o nada) |
| **Transaction Log** | Cada operación queda registrada en `_delta_log/` |
| **Time Travel** | Puedes leer versiones anteriores de la tabla |
| **Schema Enforcement** | Rechaza escrituras que no cumplan el esquema |
| **OPTIMIZE / ZORDER** | Compacta archivos pequeños y mejora performance |

En esta sección exploraremos estas capacidades sobre la tabla Silver.

In [ ]:
# =============================================================
# 4.1 — TRANSACTION LOG: DESCRIBE HISTORY
# Cada escritura crea una nueva versión registrada en el log
# =============================================================

from delta.tables import DeltaTable

# Cargar la tabla Silver como objeto DeltaTable
silver_dt = DeltaTable.forPath(spark, f"{SILVER_PATH}/ventas/")

print("=== Historial de la tabla Silver ===")
display(
    silver_dt.history()
    .select("version", "timestamp", "operation", "operationParameters", "numOutputRows")
)

In [ ]:
# =============================================================
# 4.2 — SIMULAR UNA ACTUALIZACIÓN PARA CREAR UNA NUEVA VERSIÓN
# Escribimos datos adicionales (append) para generar la versión 1
# =============================================================

# Tomar 50 registros adicionales de Silver y agregarlos (simula un lote nuevo)
df_nuevos = df_silver.limit(50).withColumn("processed_at", F.current_timestamp())

(
    df_nuevos
    .write
    .format("delta")
    .mode("append")    # append = agrega sin borrar versión anterior
    .save(f"{SILVER_PATH}/ventas/")
)

print("Append completado. Versiones disponibles ahora:")
display(
    silver_dt.history()
    .select("version", "timestamp", "operation")
)

In [ ]:
# =============================================================
# 4.3 — TIME TRAVEL: LEER UNA VERSIÓN ANTERIOR
# Esto es posible gracias al transaction log de Delta Lake
# =============================================================

# Leer la versión actual (más reciente)
df_version_actual = spark.read.format("delta").load(f"{SILVER_PATH}/ventas/")

# Leer la versión 0 (antes del append)
df_version_0 = (
    spark.read
    .format("delta")
    .option("versionAsOf", 0)   # <-- time travel por número de versión
    .load(f"{SILVER_PATH}/ventas/")
)

print(f"Registros en version actual (v1): {df_version_actual.count():,}")
print(f"Registros en version 0         : {df_version_0.count():,}")
print(f"Diferencia (registros nuevos)  : {df_version_actual.count() - df_version_0.count():,}")

# También se puede hacer time travel por timestamp:
# spark.read.format("delta").option("timestampAsOf", "2024-01-01").load(...)

In [ ]:
# =============================================================
# 4.4 — OPTIMIZE: COMPACTAR ARCHIVOS PEQUEÑOS
# En una tabla real con muchos appends se generan archivos pequeños.
# OPTIMIZE los fusiona en archivos más grandes (mejor performance).
# =============================================================

# OPTIMIZE compacta los archivos Parquet de la tabla
spark.sql(f"OPTIMIZE delta.`{SILVER_PATH}/ventas/`")

print("OPTIMIZE completado.")

# Verificar el historial después del OPTIMIZE
display(
    silver_dt.history()
    .select("version", "timestamp", "operation", "operationMetrics")
    .limit(5)
)

In [ ]:
# =============================================================
# 4.5 — CONSULTAS SQL SOBRE TABLAS DELTA
# Delta Lake soporta SQL completo, incluyendo DML (UPDATE, DELETE, MERGE)
# =============================================================

# Registrar tablas Gold como vistas temporales
spark.read.format("delta").load(f"{GOLD_PATH}/ventas_categoria_mes/").createOrReplaceTempView("gold_cat_mes")
spark.read.format("delta").load(f"{GOLD_PATH}/top_clientes/").createOrReplaceTempView("gold_clientes")

# Consulta 1: Top 3 categorías por ingresos netos totales
print("=== Top 3 categorias por ingresos totales ===")
spark.sql("""
    SELECT category,
           SUM(num_transacciones)  AS transacciones,
           SUM(ingresos_netos)     AS ingresos_netos_total
    FROM gold_cat_mes
    GROUP BY category
    ORDER BY ingresos_netos_total DESC
    LIMIT 3
""").show()

# Consulta 2: Distribución de segmentos de clientes
print("=== Distribucion por segmento ===")
spark.sql("""
    SELECT segmento,
           COUNT(*)                    AS num_clientes,
           ROUND(AVG(gasto_total), 2)  AS gasto_promedio
    FROM gold_clientes
    GROUP BY segmento
    ORDER BY gasto_promedio DESC
""").show()

---
## Parte 5 — Ejercicios integradores

Completa los siguientes ejercicios usando los datos de las capas Silver y Gold.  
Puedes usar tanto la API de DataFrames como Spark SQL.

> **Tip:** recuerda que `df_silver` ya está cargado. También puedes usar `spark.read.format("delta").load(...)` para releer cualquier capa.

**E1** — ¿Cuál es el producto más vendido (en unidades totales) en cada categoría? Muestra: categoría, producto y total de unidades.

In [ ]:
# Tu respuesta aquí


**E2** — Calcula la **tasa de descuento promedio** y los **ingresos netos por ciudad** (`store_city`). Ordena de mayor a menor ingreso.

In [ ]:
# Tu respuesta aquí


**E3** — Construye una nueva tabla Gold llamada `gold_tendencia_semanal` que muestre por semana del año: número de transacciones, ingresos netos y ticket promedio. Guárdala en Delta en `{GOLD_PATH}/tendencia_semanal/`.

> Pista: usa `F.weekofyear("transaction_date")` para extraer la semana.

In [ ]:
# Tu respuesta aquí


**E4 (Avanzado)** — Usa el **time travel de Delta Lake** para responder: ¿cuánto cambió el total de ingresos netos entre la versión 0 y la versión actual de la tabla Silver? Muestra ambos valores y la diferencia.

In [ ]:
# Tu respuesta aquí


---
## Soluciones de referencia

Descomenta cada bloque para ver la solución.

In [ ]:
# --- E1 ---
# from pyspark.sql import Window
# w = Window.partitionBy("category").orderBy(F.col("total_unidades").desc())
#
# (
#     df_silver
#     .groupBy("category", "product_name")
#     .agg(F.sum("quantity").alias("total_unidades"))
#     .withColumn("rank", F.rank().over(w))
#     .filter(F.col("rank") == 1)
#     .drop("rank")
#     .orderBy("category")
#     .show()
# )

In [ ]:
# --- E2 ---
# (
#     df_silver
#     .groupBy("store_city")
#     .agg(
#         F.round(F.avg("discount_pct") * 100, 1).alias("descuento_promedio_pct"),
#         F.round(F.sum("total_neto"), 2).alias("ingresos_netos"),
#         F.count("transaction_id").alias("num_transacciones")
#     )
#     .orderBy(F.col("ingresos_netos").desc())
#     .show()
# )

In [ ]:
# --- E3 ---
# gold_semanal = (
#     df_silver
#     .withColumn("week", F.weekofyear("transaction_date"))
#     .groupBy("year", "week")
#     .agg(
#         F.count("transaction_id").alias("num_transacciones"),
#         F.round(F.sum("total_neto"), 2).alias("ingresos_netos"),
#         F.round(F.avg("total_neto"), 2).alias("ticket_promedio")
#     )
#     .orderBy("year", "week")
# )
# gold_semanal.write.format("delta").mode("overwrite").save(f"{GOLD_PATH}/tendencia_semanal/")
# display(gold_semanal)

In [ ]:
# --- E4 ---
# ingresos_v0 = (
#     spark.read.format("delta").option("versionAsOf", 0).load(f"{SILVER_PATH}/ventas/")
#     .agg(F.round(F.sum("total_neto"), 2).alias("ingresos_v0"))
#     .collect()[0]["ingresos_v0"]
# )
# ingresos_actual = (
#     spark.read.format("delta").load(f"{SILVER_PATH}/ventas/")
#     .agg(F.round(F.sum("total_neto"), 2).alias("ingresos_actual"))
#     .collect()[0]["ingresos_actual"]
# )
# print(f"Ingresos v0      : ${ingresos_v0:>12,.2f}")
# print(f"Ingresos actual  : ${ingresos_actual:>12,.2f}")
# print(f"Diferencia       : ${ingresos_actual - ingresos_v0:>12,.2f}")

---
## Limpieza (opcional)

Ejecuta esta celda al finalizar si quieres liberar el espacio usado en DBFS.

In [ ]:
# Eliminar todos los datos del taller del DBFS
dbutils.fs.rm(BASE_PATH, recurse=True)
print(f"Directorio {BASE_PATH} eliminado correctamente.")